In [2]:
import os
import requests
import tiktoken


# Initialize
tokenizer = tiktoken.get_encoding("gpt2")

# read the file
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

# encode the text using our BPE tokenizer
enc_text = tokenizer.encode(raw_text)
print(len(enc_text))

5145


Next we remove first 50 tokens for fun

In [4]:
enc_sample = enc_text[50:]

Next we create input-target pairs

In [5]:
context_size = 4         # The context size determines how many tokens are included in the input.
x = enc_sample[:context_size]
y = enc_sample[1:context_size+1]
print(f"x: {x}")
print(f"y:      {y}")

x: [290, 4920, 2241, 287]
y:      [4920, 2241, 287, 257]


By processing the inputs along with the targets, which are the inputs shifted by one position, 
we can create the <b> next-word prediction tasks </b> 

In [6]:
for i in range(1, context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]
    print(context, "---->", desired)

[290] ----> 4920
[290, 4920] ----> 2241
[290, 4920, 2241] ----> 287
[290, 4920, 2241, 287] ----> 257


In [7]:
for i in range(1, context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]
    print(tokenizer.decode(context), "---->", tokenizer.decode([desired]))

 and ---->  established
 and established ---->  himself
 and established himself ---->  in
 and established himself in ---->  a


In [12]:
import torch

from torch.utils.data import Dataset, DataLoader
from EfficientDataLoader import GPTDatasetV1


def create_dataloader_v1(txt, batch_size=4, max_length=256,
                         stride=128, shuffle=True, drop_last=True,
                         num_workers=0):
    # Initializes the tokenizer
    tokenizer = tiktoken.get_encoding("gpt2")    
    
    # create dataset
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)  

    # drop_last=True 
    # drops the last batch if it is shorter than the specified batch_size to prevent loss spikes during training.
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,     
        num_workers=num_workers     # The number of CPU processes to use for preprocessing
    )

    return dataloader

Let’s test the dataloader with a batch size of 1 for an LLM with a context size of 4 to develop an intuition of how the GPTDatasetV1

In [21]:
dataloader = create_dataloader_v1(
    raw_text, batch_size=1, max_length=4, stride=1, shuffle=False)
data_iter = iter(dataloader)      #1

inputs, targets = next(data_iter)
print("Inputs:\n", inputs)
print("\nTargets:\n", targets)

Inputs:
 tensor([[  40,  367, 2885, 1464]])

Targets:
 tensor([[ 367, 2885, 1464, 1807]])


In [22]:
inputs, targets = next(data_iter)
print("Inputs:\n", inputs)
print("\nTargets:\n", targets)

Inputs:
 tensor([[ 367, 2885, 1464, 1807]])

Targets:
 tensor([[2885, 1464, 1807, 3619]])


To develop more intuition for how the data loader works, try to run it with different settings such as max_length=2 and stride=2, and max_length=8 and stride=2.

In [26]:
dataloader2 = create_dataloader_v1(
    raw_text, batch_size=1, max_length=2, stride=2, shuffle=False)
data_iter = iter(dataloader2)     
inputs, targets = next(data_iter)
print("Inputs:\n", inputs)
print("\nTargets:\n", targets)

inputs, targets = next(data_iter)
print("\nSecond Batch:")
print("Inputs:\n", inputs)
print("\nTargets:\n", targets)

Inputs:
 tensor([[ 40, 367]])

Targets:
 tensor([[ 367, 2885]])

Second Batch:

Inputs:
 tensor([[2885, 1464]])

Targets:
 tensor([[1464, 1807]])


In [29]:
dataloader3 = create_dataloader_v1(
    raw_text, batch_size=1, max_length=8, stride=2, shuffle=False)
data_iter = iter(dataloader3)   

inputs, targets = next(data_iter)
print("Inputs:\n", inputs)
print("\nTargets:\n", targets)

inputs, targets = next(data_iter)
print("\nSecond Batch:")
print("Inputs:\n", inputs)
print("\nTargets:\n", targets)
print("\nInputs shape:\n", inputs.shape)

Inputs:
 tensor([[  40,  367, 2885, 1464, 1807, 3619,  402,  271]])

Targets:
 tensor([[  367,  2885,  1464,  1807,  3619,   402,   271, 10899]])

Second Batch:
Inputs:
 tensor([[ 2885,  1464,  1807,  3619,   402,   271, 10899,  2138]])

Targets:
 tensor([[ 1464,  1807,  3619,   402,   271, 10899,  2138,   257]])

Inputs shape:
 torch.Size([1, 8])


In [30]:
dataloader = create_dataloader_v1(
    raw_text, batch_size=8, max_length=4, stride=4,
    shuffle=False
)

data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print("Inputs:\n", inputs)
print("\nTargets:\n", targets)
print("\nInputs shape:\n", inputs.shape)

Inputs:
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])

Targets:
 tensor([[  367,  2885,  1464,  1807],
        [ 3619,   402,   271, 10899],
        [ 2138,   257,  7026, 15632],
        [  438,  2016,   257,   922],
        [ 5891,  1576,   438,   568],
        [  340,   373,   645,  1049],
        [ 5975,   284,   502,   284],
        [ 3285,   326,    11,   287]])

Inputs shape:
 torch.Size([8, 4])


As we can see, the token ID tensor is 8 × 4 dimensional, meaning that the data batch consists of eight text samples with four tokens each.

### Create Word Embeddings

In [31]:
input_ids = torch.tensor([2, 3, 5, 1])
vocab_size = 6
output_dim = 3

torch.manual_seed(123)
embedding_layer = torch.nn.Embedding(vocab_size, output_dim)
print(embedding_layer.weight)

Parameter containing:
tensor([[ 0.3374, -0.1778, -0.1690],
        [ 0.9178,  1.5810,  1.3010],
        [ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096]], requires_grad=True)


In [32]:
print(embedding_layer(torch.tensor([3])))

tensor([[-0.4015,  0.9666, -1.1481]], grad_fn=<EmbeddingBackward0>)


In [34]:
vocab_size = 50257
output_dim = 256
token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)
token_embeddings = token_embedding_layer(inputs)
print(token_embeddings.shape)

torch.Size([8, 4, 256])


<b> The 8 × 4 × 256–dimensional tensor output shows that each token ID is now embedded as a 256-dimensional vector. </b>

#### GPT Absolute Embedding Approach

In [35]:
context_length = max_length = 4
pos_embedding_layer = torch.nn.Embedding(context_length, output_dim)
pos_embeddings = pos_embedding_layer(torch.arange(context_length))
print(pos_embeddings.shape)

torch.Size([4, 256])


As we can see, the positional embedding tensor consists of four 256-dimensional vectors

We can now add these directly to the token embeddings, where PyTorch will add the 4 × 256–dimensional pos_embeddings tensor to each 4 × 256–dimensional token embedding tensor in each of the eight batches

In [36]:
input_embeddings = token_embeddings + pos_embeddings
print(input_embeddings.shape)

torch.Size([8, 4, 256])


In [38]:
print(token_embedding_layer(torch.tensor([3])))

tensor([[-1.1071, -0.9817, -0.6754, -0.4628,  1.4323,  0.2217,  0.8599, -1.0025,
         -0.8137, -0.3211,  0.7578,  0.4713,  0.4612,  0.9181,  0.5303,  0.2747,
          1.0646, -0.5763, -0.7217, -0.5066,  1.4247,  0.0735,  0.6832, -0.4913,
         -0.2808, -1.3415, -2.3031, -0.3688,  0.6496, -0.7473, -0.5815, -0.7616,
          0.8824,  0.0963,  0.4593, -0.5073,  0.1822,  0.3349, -0.4123, -1.3619,
          0.6505, -0.3008, -0.3973, -0.3082, -0.0115, -1.2135,  1.4970, -0.6437,
         -0.4811,  0.0594, -0.0220,  1.8075,  0.5184, -0.5878,  1.1037, -0.4102,
         -1.1145,  1.0062, -1.5628,  0.5647, -0.4338, -1.4368,  0.1148, -0.6997,
          0.7271, -0.4871,  0.5698,  1.3348,  1.4644,  1.0372, -1.1234,  2.2164,
         -1.1628, -0.7375, -0.2083,  0.3547,  1.1037, -0.6872,  0.5159,  0.4956,
          0.8687,  0.0246, -0.2786, -0.7115,  0.6576,  0.4118,  0.1366,  1.3669,
          0.1136,  1.8179,  1.0588, -0.2979, -1.0573, -0.6687,  0.0040,  0.8387,
         -1.6399,  1.3014,  